# Task 9: RaSP 真结构 ΔΔG

**方法**: 使用 KULL-Centre 的 RaSP (ML-ddG-Blaabjerg 2022) —— 3D-CNN 空腔模型 + 下游 ΔΔG 回归。

**Pipeline**:
1. PDB 清洗 (pdbfixer 加氢 + 修缺失原子) → `_clean.pdb`
2. 提取突变位点的局部原子环境 → 6 类原子 × 18³ 网格
3. CavityModel (3D-CNN) → 100 维隐变量
4. DownstreamModel (10 个 ensemble) → ΔΔG (Fermi 变换后反解)

**依赖**: `openmm`, `pdbfixer` (已安装); 预训练权重在 `pretrained_models/`。

In [5]:
import os, sys, re, time, warnings, json
import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

warnings.filterwarnings("ignore")

# ===== 路径 =====
BASE_PATH   = "/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/"
PDB_DIR     = "/mnt/volume6/czj/labLGN/LabLZ/models/alphafold_pdb/"
RASP_ROOT   = "/mnt/volume6/czj/labLGN/LabLZ/models/_2022_ML-ddG-Blaabjerg-main/"

# 将 RaSP src 加入 path
sys.path.insert(0, RASP_ROOT + "src")
sys.path.insert(0, RASP_ROOT + "src/pdb_parser_scripts")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import Bio.PDB
from Bio.PDB.PDBParser import PDBParser
from Bio.PDB.SASA import ShrakeRupley
from Bio.PDB.Polypeptide import index_to_one, one_to_index, three_to_index
import pdbfixer

# 注意: 不 import simtk (改用 openmm 直接)
import openmm.app as openmm_app

# RaSP 模型
from rasp_model import CavityModel, DownstreamModel, ResidueEnvironment, ResidueEnvironmentsDataset, ToTensor
from helpers import inverse_fermi_transform, populate_dfs_with_resenvs, ds_pred

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"RaSP root: {RASP_ROOT}")

# ===== 加载预训练模型 =====
# Cavity model
cavity_model = CavityModel(get_latent=True).to(DEVICE)
cavity_model.load_state_dict(
    torch.load(RASP_ROOT + "pretrained_models/cavity/model.pt", map_location=DEVICE))
cavity_model.eval()
print("CavityModel 已加载")

# Downstream model (template, weights will be loaded per ensemble member)
ds_model = DownstreamModel().to(DEVICE)
ds_model.apply(lambda m: nn.init.kaiming_uniform_(m.weight) if isinstance(m, nn.Linear) else None)

NUM_ENSEMBLE = 10
print(f"DownstreamModel 模板已加载 (ensemble={NUM_ENSEMBLE})")

Device: cuda
RaSP root: /mnt/volume6/czj/labLGN/LabLZ/models/_2022_ML-ddG-Blaabjerg-main/
CavityModel 已加载
DownstreamModel 模板已加载 (ensemble=10)


## 9a. PDB 清洗 (pdbfixer 加氢)

用 pdbfixer 替代 reduce：修复非标准残基 → 补缺失原子 → 加氢。

In [8]:
import tempfile

PDBIO = Bio.PDB.PDBIO()
PDB_PARSER = Bio.PDB.PDBParser(PERMISSIVE=1)

class NonHetSelector(Bio.PDB.Select):
    """保留标准残基，取第一个替代构象"""
    def accept_residue(self, residue):
        resname = residue.get_resname().strip()
        try:
            three_to_index(resname)
            return True
        except:
            return resname in pdbfixer.pdbfixer.substitutions

    def accept_atom(self, atom):
        return (
            not atom.is_disordered()
            or atom.get_altloc() == "A"
            or atom.get_altloc() == "1"
        ) and atom.id[0] in ["C", "H", "N", "O", "S", "P"]


def clean_pdb_simple(input_pdb_path, output_pdb_path):
    """
    简化版 PDB 清洗: NonHetSelector → pdbfixer(加氢)。
    不使用 reduce，直接用 pdbfixer 的 addMissingHydrogens。
    """
    pdbid = os.path.basename(input_pdb_path).split(".")[0]
    
    # Step 1: 解析 + NonHetSelector 过滤
    structure = PDB_PARSER.get_structure(pdbid, input_pdb_path)
    first_model = structure[0]
    
    # Step 2: 写临时文件 → pdbfixer
    with tempfile.NamedTemporaryFile(mode="wt", suffix=".pdb", delete=False) as tmp:
        tmp_path = tmp.name
    
    try:
        # 替换 altloc
        for chain in first_model:
            for res in chain:
                for atom in res:
                    atom.set_altloc(" ")
        
        PDBIO.set_structure(first_model)
        PDBIO.save(tmp_path, select=NonHetSelector())
        
        # pdbfixer
        fixer = pdbfixer.PDBFixer(tmp_path)
        fixer.findMissingResidues()
        fixer.findNonstandardResidues()
        fixer.replaceNonstandardResidues()
        fixer.findMissingAtoms()
        fixer.addMissingAtoms()
        fixer.addMissingHydrogens(7.0)
        
        # 保存 cleaned PDB
        with open(output_pdb_path, "w") as f:
            openmm_app.PDBFile.writeFile(fixer.topology, fixer.positions, f, keepIds=True)
        
        return True
    except Exception as e:
        return False, str(e)[:80]
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)

print("PDB 清洗函数就绪")

PDB 清洗函数就绪


## 9b. 提取突变位点的原子环境 (ResidueEnvironment)

模仿 RaSP 的 `extract_environments.py`，但用 `ShrakeRupley` 替代 DSSP 算 SASA，
直接从 cleaned PDB 读 B-factor (pLDDT)。

In [9]:
# 与 RaSP 一致的原子类型映射 (取 atom name 首字母)
ATOM_TYPE_LIST = ["C", "N", "O", "H", "S", "P"]
ATOM_TYPE_MAP = {k: i for i, k in enumerate(ATOM_TYPE_LIST)}

AMINO_ACIDS_3TO1 = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C',
    'GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I',
    'LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P',
    'SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V',
    'MSE':'M',
}

MAX_RADIUS = 9.0  # RaSP 默认: 9 Å


def define_coordinate_system(pos_N, pos_CA, pos_C):
    """RaSP 风格: 以 CA 为原点，N-C 为 e1，侧链方向为 e2，e3 = e1 × e2"""
    import Bio.PDB.vectors
    e1 = pos_C - pos_N
    e1 = e1 / np.linalg.norm(e1)
    # 估算 CB 位置 (绕 CA-C 轴旋转 N 原子 120°)
    pos_N_res = pos_N - pos_CA
    axis = pos_CA - pos_C
    pos_CB = np.dot(
        Bio.PDB.rotaxis((120.0 / 180.0) * np.pi, Bio.PDB.vectors.Vector(axis)),
        pos_N_res,
    )
    e2 = pos_CB / np.linalg.norm(pos_CB)
    e3 = np.cross(e1, e2)
    e2 = np.cross(e1, -e3)  # 修正正交性
    # Z-direction = outward (e3), RaSP 默认
    rot_matrix = np.array([e1, e2, e3])
    return rot_matrix


def extract_resenv_from_pdb(clean_pdb_path, pdb_resnum, chain_id="A"):
    """
    从 cleaned PDB 提取 RaSP 风格的 ResidueEnvironment（空腔模式）。
    
    关键: 提取目标残基周围 9Å 内的所有原子，EXCLUDING 目标残基本身。
    坐标系: 以 CA 为原点，N-C 为 e1 轴，侧链方向为 e2。
    """
    pdbid = os.path.basename(clean_pdb_path).split(".")[0].replace("_clean", "")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdbid, clean_pdb_path)
    model = structure[0]
    
    # ===== 1. 收集所有原子的全局信息 =====
    all_coords = []       # (n_total, 3)
    all_atom_names = []   # str
    all_res_indices = []  # 每个原子属于哪个残基 (global index)
    target_res_global_idx = None  # 目标残基的 global index
    
    global_res_idx = 0
    for chain in model:
        for res in chain:
            if res.get_id()[0] != " ":
                continue
            for atom in res:
                aname = atom.get_name().strip()
                all_atom_names.append(aname)
                all_coords.append(atom.get_coord())
                all_res_indices.append(global_res_idx)
            # 检查是否为目标残基
            if (chain.id == chain_id or chain_id is None) and res.get_id()[1] == pdb_resnum:
                target_res_global_idx = global_res_idx
                target_res = res
                target_chain = chain
            global_res_idx += 1
    
    if target_res_global_idx is None:
        # 按索引 fallback
        global_res_idx = 0
        for chain in model:
            residues_list = [r for r in chain if r.get_id()[0] == " "]
            if pdb_resnum <= len(residues_list):
                target_res_global_idx = global_res_idx + pdb_resnum - 1
                target_res = residues_list[pdb_resnum - 1]
                break
            global_res_idx += len(residues_list)
        else:
            return None, "pos_not_found"
    
    if not all_coords:
        return None, "no_atoms_in_pdb"
    
    all_coords = np.array(all_coords, dtype=np.float32)
    all_atom_names = np.array(all_atom_names, dtype=str)
    all_res_indices = np.array(all_res_indices, dtype=np.int32)
    
    # ===== 2. 提取 N, CA, C 用于定义局部坐标系 =====
    n_coord = ca_coord = c_coord = None
    for atom in target_res:
        aname = atom.get_name().strip()
        if aname == "N":
            n_coord = atom.get_coord()
        elif aname == "CA":
            ca_coord = atom.get_coord()
        elif aname == "C":
            c_coord = atom.get_coord()
    
    if n_coord is None or ca_coord is None or c_coord is None:
        return None, "missing_backbone_atoms"
    
    # ===== 3. 定义局部坐标系 + 转换所有原子坐标 =====
    rot_matrix = define_coordinate_system(n_coord, ca_coord, c_coord)
    xyz_relative = all_coords - ca_coord  # 以 CA 为原点
    xyz_local = np.dot(rot_matrix, xyz_relative.T).T  # (n_total, 3)
    
    # ===== 4. 筛选: 半径 < 9Å 且排除目标残基本身 =====
    r = np.sqrt(xyz_local[:, 0]**2 + xyz_local[:, 1]**2 + xyz_local[:, 2]**2)
    mask = np.logical_and(r < MAX_RADIUS, all_res_indices != target_res_global_idx)
    selector = np.where(mask)[0]
    
    if len(selector) == 0:
        return None, "no_atoms_in_cavity"
    
    # ===== 5. 提取选中的原子 =====
    selected_coords = xyz_local[selector]
    selected_names = all_atom_names[selector]
    
    # 原子类型: 取 atom name 首字母
    atom_types = np.array([
        ATOM_TYPE_MAP.get(name[0], -1) for name in selected_names
    ], dtype=np.int32)
    valid = atom_types >= 0
    selected_coords = selected_coords[valid]
    atom_types = atom_types[valid]
    
    if len(atom_types) == 0:
        return None, "no_valid_atom_types"
    
    # ===== 6. SASA + B-factor =====
    b_factor = 0.0
    for atom in target_res:
        b = atom.get_bfactor()
        if b > 0:
            b_factor = max(b_factor, b)  # 用最大值(pLDDT)
    if b_factor == 0:
        b_factor = 90.0
    
    try:
        ShrakeRupley().compute(structure, level="R")
        sasa = target_res.sasa
    except Exception:
        sasa = 0.0
    
    # ===== 7. 残基类型 one-hot =====
    resname = target_res.get_resname().strip()
    aa_1 = AMINO_ACIDS_3TO1.get(resname, "X")
    try:
        res_idx = one_to_index(aa_1)
    except:
        res_idx = 20
    restype_onehot = np.zeros(21, dtype=np.float32)
    restype_onehot[res_idx] = 1.0
    
    resenv = ResidueEnvironment(
        xyz_coords=selected_coords.astype(np.float32),
        atom_types=atom_types,
        sasa=sasa,
        b_factors=b_factor,
        restype_onehot=restype_onehot,
        chain_id=chain_id if chain_id else target_chain.id,
        pdb_residue_number=pdb_resnum,
        pdb_id=pdbid
    )
    return resenv, "ok"

print("ResidueEnvironment (cavity 模式) 提取函数就绪")

ResidueEnvironment (cavity 模式) 提取函数就绪


## 9c. 小样本测试 (5 个变体，完整 pipeline)

测试: PDB 清洗 → 环境提取 → RaSP 推理 → ΔΔG。

In [6]:
# ===== 加载数据，取前 5 个有 PDB 的变体 =====
df_main = pd.read_csv("/mnt/volume6/czj/labLGN/LabLZ/data_preparation/cell2024_model_single_subst.csv")

def parse_mutation(mut_str):
    if not isinstance(mut_str, str): return None, None, None
    m = re.match(r'^([A-Z])(\d+)([A-Z])$', mut_str.strip())
    return (m.group(1), int(m.group(2)), m.group(3)) if m else (None, None, None)

# 筛选有 PDB 的行
df_main["_has_pdb"] = df_main["Uniprot"].apply(
    lambda u: os.path.exists(PDB_DIR + f"{u}.pdb") if isinstance(u, str) else False)
df_pool = df_main[df_main["_has_pdb"]].copy()
df_pool["wt_aa"], df_pool["pos"], df_pool["mt_aa"] = zip(
    *df_pool["Mutation_used"].apply(parse_mutation))

# 取前 5 个
df_test = df_pool.head(5).reset_index(drop=True)
print(f"测试样本 (前5个):")
for _, row in df_test.iterrows():
    pdb = PDB_DIR + f"{row['Uniprot']}.pdb"
    print(f"  {row['Gene']} {row['Mutation_used']}  Uniprot={row['Uniprot']}  PDB_exists={os.path.exists(pdb)}")

测试样本 (前5个):
  LDHA K222E  Uniprot=P00338  PDB_exists=True
  PTEN K6N  Uniprot=P60484  PDB_exists=True
  ASS1 K310Q  Uniprot=P00966  PDB_exists=True
  EPHX2 K55R  Uniprot=P34913  PDB_exists=True
  FTH1 K54R  Uniprot=P02794  PDB_exists=True


In [ ]:
# ===== 清洗 PDB (仅测试样本) =====
CLEAN_DIR = "/mnt/volume6/czj/labLGN/LabLZ/models/alphafold_pdb_clean/"
os.makedirs(CLEAN_DIR, exist_ok=True)

print("Step 1: PDB 清洗 (pdbfixer 加氢)...")
t0 = time.time()
uniprot_done = set()
for _, row in df_test.iterrows():
    uid = row["Uniprot"]
    if uid in uniprot_done:
        continue
    
    input_pdb = PDB_DIR + f"{uid}.pdb"
    output_pdb = CLEAN_DIR + f"{uid}_clean.pdb"
    
    if os.path.exists(output_pdb):
        print(f"  {uid}: 已有 cleaned PDB, 跳过")
    else:
        result = clean_pdb_simple(input_pdb, output_pdb)
        if result is True:
            print(f"  {uid}: OK")
        else:
            print(f"  {uid}: FAILED - {result[1]}")
    uniprot_done.add(uid)

print(f"清洗完毕, 耗时 {time.time()-t0:.1f}s")

Step 1: PDB 清洗 (pdbfixer 加氢)...
  P00338: 已有 cleaned PDB, 跳过
  P60484: 已有 cleaned PDB, 跳过
  P00966: 已有 cleaned PDB, 跳过
  P34913: 已有 cleaned PDB, 跳过
  P02794: 已有 cleaned PDB, 跳过
清洗完毕, 耗时 0.0s


In [12]:
# ===== 提取 ResidueEnvironment 并运行 RaSP 推理 =====

print("\nStep 2: 提取原子环境 + RaSP 推理...")

# 准备 PDB 频率 (用于下游模型输入)
pdb_freq_path = RASP_ROOT + "data/train/cavity/pdb_frequencies.npz"
if os.path.exists(pdb_freq_path):
    pdb_nlfs = -np.log(np.load(pdb_freq_path)["frequencies"])
else:
    # Fallback: 均匀频率
    pdb_nlfs = -np.log(np.ones(20) / 20)
    print("⚠ pdb_frequencies.npz 未找到，使用均匀频率")

to_tensor = ToTensor(DEVICE)
results_test = []

for i, (_, row) in enumerate(df_test.iterrows()):
    uid = row["Uniprot"]
    wt_aa = row["wt_aa"]
    pos = row["pos"]
    mt_aa = row["mt_aa"]
    
    clean_pdb = CLEAN_DIR + f"{uid}_clean.pdb"
    if not os.path.exists(clean_pdb):
        results_test.append({"Gene": row["Gene"], "Mutation_used": row["Mutation_used"],
                            "ddg_rasp": np.nan, "status": "no_clean_pdb"})
        continue
    
    # 提取 WT 环境
    resenv, status = extract_resenv_from_pdb(clean_pdb, pos)
    if resenv is None:
        results_test.append({"Gene": row["Gene"], "Mutation_used": row["Mutation_used"],
                            "ddg_rasp": np.nan, "status": status})
        continue
    
    # 验证 PDB 中该位置的氨基酸与 WT 一致
    pdb_aa_idx = np.argmax(resenv.restype_onehot)
    if pdb_aa_idx < 20:
        pdb_aa = index_to_one(pdb_aa_idx)
        if pdb_aa != wt_aa:
            results_test.append({"Gene": row["Gene"], "Mutation_used": row["Mutation_used"],
                                "ddg_rasp": np.nan,
                                "status": f"wt_mismatch(pdb={pdb_aa},label={wt_aa})"})
            continue
    
    # 构建下游模型输入
    wt_idx = one_to_index(wt_aa)
    mt_idx = one_to_index(mt_aa)
    wt_nlf = pdb_nlfs[wt_idx]
    mt_nlf = pdb_nlfs[mt_idx]
    
    x_ds = np.hstack([
        np.eye(20)[wt_idx],  # WT one-hot
        np.eye(20)[mt_idx],  # MT one-hot
        [wt_nlf],            # WT frequency
        [mt_nlf]             # MT frequency
    ]).astype(np.float32)
    
    # Cavity model 推理
    # ToTensor 返回 {"x_": (n_atoms, 4)} where cols = [atom_type, x, y, z]
    # CavityModel 需要 (n_atoms, 5) where cols = [env_id, atom_type, x, y, z]
    sample_tensor = to_tensor(resenv)
    env_id = torch.zeros(sample_tensor["x_"].shape[0], 1, dtype=torch.float32).to(DEVICE)
    x_cavity_full = torch.cat([env_id, sample_tensor["x_"]], dim=1)  # (n_atoms, 5)
    
    with torch.no_grad():
        latent = cavity_model(x_cavity_full)  # (1, 100)
    
    # 10 个 ensemble 下游模型
    ddg_fermi_preds = []
    for ens_i in range(NUM_ENSEMBLE):
        ds_path = RASP_ROOT + f"pretrained_models/ds/ds_model_{ens_i}/model.pt"
        if os.path.exists(ds_path):
            ds_model.load_state_dict(torch.load(ds_path, map_location=DEVICE))
            ds_model.eval()
            x_combined = torch.cat([latent, torch.tensor(x_ds).unsqueeze(0).to(DEVICE)], dim=1)
            with torch.no_grad():
                ddg_fermi = ds_model(x_combined).item()
            ddg_fermi_preds.append(ddg_fermi)
    
    if ddg_fermi_preds:
        ddg_fermi_median = np.median(ddg_fermi_preds)
        ddg = inverse_fermi_transform(ddg_fermi_median)
    else:
        ddg = np.nan
    
    results_test.append({
        "Gene": row["Gene"],
        "Mutation_used": row["Mutation_used"],
        "Uniprot": uid,
        "wt_aa": wt_aa, "pos": pos, "mt_aa": mt_aa,
        "ddg_rasp": ddg,
        "status": "ok"
    })
    print(f"  [{i+1}/5] {row['Gene']} {row['Mutation_used']}: ddg_rasp={ddg:.3f}")

# 展示
df_r = pd.DataFrame(results_test)
print(f"\n测试结果:")
print(df_r[["Gene", "Mutation_used", "wt_aa", "pos", "mt_aa", "ddg_rasp", "status"]].to_string(index=False))
n_ok = (df_r["status"] == "ok").sum()
print(f"\n成功: {n_ok}/{len(df_r)}")
if n_ok > 0:
    ddg_vals = df_r[df_r["status"] == "ok"]["ddg_rasp"]
    print(f"ddg_rasp 统计: mean={ddg_vals.mean():.3f}, range=[{ddg_vals.min():.3f}, {ddg_vals.max():.3f}]")


Step 2: 提取原子环境 + RaSP 推理...
  [1/5] LDHA K222E: ddg_rasp=0.549
  [2/5] PTEN K6N: ddg_rasp=1.449
  [3/5] ASS1 K310Q: ddg_rasp=0.355
  [4/5] EPHX2 K55R: ddg_rasp=0.590
  [5/5] FTH1 K54R: ddg_rasp=-0.001

测试结果:
 Gene Mutation_used wt_aa  pos mt_aa  ddg_rasp status
 LDHA         K222E     K  222     E  0.549203     ok
 PTEN           K6N     K    6     N  1.449393     ok
 ASS1         K310Q     K  310     Q  0.354705     ok
EPHX2          K55R     K   55     R  0.589973     ok
 FTH1          K54R     K   54     R -0.001109     ok

成功: 5/5
ddg_rasp 统计: mean=0.588, range=[-0.001, 1.449]


## 9d. 全量计算（用户自行运行）

**⚠ 以下为全量 2179 变体的完整 pipeline:**
1. 清洗所有 AlphaFold PDB (约 910 个) — 预计 30-60 分钟
2. 提取所有突变位点的 ResidueEnvironment — 预计 5-10 分钟
3. RaSP 推理 — 预计 5-10 分钟

**请在确认小样本测试通过后，将 RUN_FULL 改为 True 并运行。**

In [13]:
RUN_FULL = True  # ← 改为 True 后运行全量

if RUN_FULL:
    print("=" * 70)
    print("全量 RaSP ΔΔG (优化版: DS预加载 + PDB分组缓存 + 断点续传)")
    print("=" * 70)
    
    # ===== 0. 断点续传 =====
    OUTPUT_CSV = BASE_PATH + "ddg_rasp.csv"
    if os.path.exists(OUTPUT_CSV):
        df_done = pd.read_csv(OUTPUT_CSV)
        done_keys = set(df_done[df_done["status"] == "ok"]["KEY"].tolist())
        print(f"已有结果: {len(done_keys)} OK, 断点续传")
    else:
        done_keys = set()
        pd.DataFrame(columns=["KEY","ddg_rasp","status"]).to_csv(OUTPUT_CSV, index=False)
    
    # ===== 1. 准备 =====
    df_all = df_main.copy()
    df_all["wt_aa"], df_all["pos"], df_all["mt_aa"] = zip(*df_all["Mutation_used"].apply(parse_mutation))
    df_all["KEY"] = df_all["Gene"] + "||" + df_all["Mutation_used"]
    df_todo = df_all[~df_all["KEY"].isin(done_keys)].copy()
    print(f"总:{len(df_all)}, 已完成:{len(done_keys)}, 待处理:{len(df_todo)}")
    
    if len(df_todo) == 0:
        print("全部完成!")
    else:
        # ===== 2. 预加载 DS 模型 (一次性, 不重复 load) =====
        print("\n预加载 10 个 DownstreamModel...")
        ds_models = []
        for ens_i in range(NUM_ENSEMBLE):
            ds_path = RASP_ROOT + f"pretrained_models/ds/ds_model_{ens_i}/model.pt"
            m = DownstreamModel().to(DEVICE)
            m.load_state_dict(torch.load(ds_path, map_location=DEVICE))
            m.eval()
            ds_models.append(m)
        print(f"  已加载 {len(ds_models)} 个")
        
        pdb_freq_path = RASP_ROOT + "data/train/cavity/pdb_frequencies.npz"
        pdb_nlfs = -np.log(np.load(pdb_freq_path)["frequencies"]) if os.path.exists(pdb_freq_path) else -np.log(np.ones(20)/20)
        
        to_tensor = ToTensor(DEVICE)
        CLEAN_DIR = BASE_PATH + "alphafold_pdb_clean/"
        SAVE_EVERY = 100
        
        # ===== 3. 按 Uniprot 分组 (同一 PDB 只解析一次) =====
        uid_groups = list(df_todo.groupby("Uniprot"))
        print(f"按 Uniprot 分组: {len(uid_groups)} 个 PDB\n")
        
        results_batch = []
        n_ok, n_processed, t0 = 0, 0, time.time()
        
        for uid, grp in uid_groups:
            clean_pdb = CLEAN_DIR + f"{uid}_clean.pdb"
            if not os.path.exists(clean_pdb):
                for _, row in grp.iterrows():
                    results_batch.append({"KEY": row["KEY"], "ddg_rasp": np.nan, "status": "no_clean_pdb"})
                n_processed += len(grp)
                continue
            
            # ---- 解析 PDB (一次) ----
            pdbid = uid
            parser = PDBParser(QUIET=True)
            structure = parser.get_structure(pdbid, clean_pdb)
            model = structure[0]
            
            all_coords, all_atom_names, all_res_indices = [], [], []
            residue_map = {}  # pdb_resnum → {global_idx, res_obj}
            gidx = 0
            for chain in model:
                for res in chain:
                    if res.get_id()[0] != " ": continue
                    rnum = res.get_id()[1]
                    residue_map[rnum] = {"global_idx": gidx, "res": res}
                    for atom in res:
                        all_atom_names.append(atom.get_name().strip())
                        all_coords.append(atom.get_coord())
                        all_res_indices.append(gidx)
                    gidx += 1
            
            if not all_coords:
                for _, row in grp.iterrows():
                    results_batch.append({"KEY": row["KEY"], "ddg_rasp": np.nan, "status": "no_atoms"})
                n_processed += len(grp)
                continue
            
            all_coords = np.array(all_coords, dtype=np.float32)
            all_atom_names = np.array(all_atom_names, dtype=str)
            all_res_indices = np.array(all_res_indices, dtype=np.int32)
            
            try: ShrakeRupley().compute(structure, level="R")
            except: pass
            
            # ---- 处理该 PDB 的所有变体 ----
            for _, row in grp.iterrows():
                wt_aa, pos, mt_aa, key = row["wt_aa"], row["pos"], row["mt_aa"], row["KEY"]
                
                rinfo = residue_map.get(pos)
                if rinfo is None:
                    results_batch.append({"KEY": key, "ddg_rasp": np.nan, "status": "pos_not_found"})
                    n_processed += 1; continue
                
                target_res, tg_idx = rinfo["res"], rinfo["global_idx"]
                
                # N,CA,C
                nc = cc = cac = None
                for atom in target_res:
                    an = atom.get_name().strip()
                    if an == "N": nc = atom.get_coord()
                    elif an == "CA": cac = atom.get_coord()
                    elif an == "C": cc = atom.get_coord()
                if nc is None:
                    results_batch.append({"KEY": key, "ddg_rasp": np.nan, "status": "missing_bb"})
                    n_processed += 1; continue
                
                # 坐标变换
                rm = define_coordinate_system(nc, cac, cc)
                xyz_l = np.dot(rm, (all_coords - cac).T).T
                r = np.sqrt(xyz_l[:,0]**2 + xyz_l[:,1]**2 + xyz_l[:,2]**2)
                idxs = np.where((r < MAX_RADIUS) & (all_res_indices != tg_idx))[0]
                if len(idxs) == 0:
                    results_batch.append({"KEY": key, "ddg_rasp": np.nan, "status": "empty_cavity"})
                    n_processed += 1; continue
                
                coords_sel = xyz_l[idxs]; names_sel = all_atom_names[idxs]
                a_types = np.array([ATOM_TYPE_MAP.get(n[0], -1) for n in names_sel], dtype=np.int32)
                v = a_types >= 0
                if v.sum() == 0:
                    results_batch.append({"KEY": key, "ddg_rasp": np.nan, "status": "no_valid_atoms"})
                    n_processed += 1; continue
                coords_sel = coords_sel[v]; a_types = a_types[v]
                
                # WT check
                pdb_aa = AMINO_ACIDS_3TO1.get(target_res.get_resname().strip(), "X")
                if pdb_aa != "X" and pdb_aa != wt_aa:
                    results_batch.append({"KEY": key, "ddg_rasp": np.nan, "status": f"wt_mismatch"})
                    n_processed += 1; continue
                
                # ResidueEnvironment
                bf = max([atom.get_bfactor() for atom in target_res if atom.get_bfactor() > 0], default=90.0)
                sasa = target_res.sasa if hasattr(target_res, 'sasa') else 0.0
                try: ri = one_to_index(pdb_aa) if pdb_aa != "X" else 20
                except: ri = 20
                oh = np.zeros(21, dtype=np.float32); oh[ri] = 1.0
                resenv = ResidueEnvironment(xyz_coords=coords_sel.astype(np.float32),
                                            atom_types=a_types, sasa=sasa, b_factors=bf,
                                            restype_onehot=oh, chain_id="A",
                                            pdb_residue_number=pos, pdb_id=uid)
                
                # x_ds
                if wt_aa not in "ARNDCQEGHILKMFPSTWYV" or mt_aa not in "ARNDCQEGHILKMFPSTWYV":
                    results_batch.append({"KEY": key, "ddg_rasp": np.nan, "status": "invalid_aa"})
                    n_processed += 1; continue
                wi, mi = one_to_index(wt_aa), one_to_index(mt_aa)
                x_ds = np.hstack([np.eye(20)[wi], np.eye(20)[mi], [pdb_nlfs[wi]], [pdb_nlfs[mi]]]).astype(np.float32)
                
                # Cavity
                st = to_tensor(resenv)
                eid = torch.zeros(st["x_"].shape[0], 1, dtype=torch.float32).to(DEVICE)
                x_cav = torch.cat([eid, st["x_"]], dim=1)
                with torch.no_grad(): latent = cavity_model(x_cav)
                
                # Ensemble (预加载, 不重复 load_state_dict!)
                f_vals = []
                x_ds_t = torch.tensor(x_ds).unsqueeze(0).to(DEVICE)
                for m_ds in ds_models:
                    f_vals.append(m_ds(torch.cat([latent, x_ds_t], dim=1)).item())
                
                ddg = inverse_fermi_transform(np.median(f_vals)) if f_vals else np.nan
                results_batch.append({"KEY": key, "ddg_rasp": ddg, "status": "ok"})
                n_ok += 1; n_processed += 1
            
            # ---- 进度 + 增量存盘 ----
            if n_processed % SAVE_EVERY < max(len(grp), 1):
                elapsed = time.time() - t0
                rate = n_processed / elapsed if elapsed > 0 else 0
                eta = (len(df_todo) - n_processed) / rate if rate > 0 else 0
                print(f"  {n_processed}/{len(df_todo)} OK={n_ok} 速率={rate:.1f}var/s ETA={eta:.0f}s", flush=True)
                
                df_batch = pd.DataFrame(results_batch)
                if os.path.exists(OUTPUT_CSV):
                    df_ex = pd.read_csv(OUTPUT_CSV)
                    ek = set(df_ex["KEY"].tolist())
                    df_batch = df_batch[~df_batch["KEY"].isin(ek)]
                    df_comb = pd.concat([df_ex, df_batch], ignore_index=True)
                else:
                    df_comb = df_batch
                df_comb.to_csv(OUTPUT_CSV, index=False)
                results_batch = []
        
        # ===== 最终存盘 =====
        if results_batch:
            df_batch = pd.DataFrame(results_batch)
            if os.path.exists(OUTPUT_CSV):
                df_ex = pd.read_csv(OUTPUT_CSV)
                ek = set(df_ex["KEY"].tolist())
                df_batch = df_batch[~df_batch["KEY"].isin(ek)]
                df_comb = pd.concat([df_ex, df_batch], ignore_index=True)
            else:
                df_comb = df_batch
            df_comb.to_csv(OUTPUT_CSV, index=False)
        
        elapsed = time.time() - t0
        df_final = pd.read_csv(OUTPUT_CSV)
        n_ok_f = (df_final["status"] == "ok").sum()
        print(f"\n完成! 耗时={elapsed:.0f}s 速率={len(df_todo)/elapsed:.1f}var/s")
        print(f"成功:{n_ok_f}/{len(df_final)} ({n_ok_f/len(df_final)*100:.1f}%)")
        print(df_final["status"].value_counts().to_string())
        ddgv = df_final[df_final["status"]=="ok"]["ddg_rasp"]
        if len(ddgv) > 0:
            print(f"ddg_rasp: mean={ddgv.mean():.3f} std={ddgv.std():.3f} min={ddgv.min():.3f} max={ddgv.max():.3f}")
else:
    print("RUN_FULL = False。预计速率: 5-10 var/s (优化后), 全量约 4-8 分钟")

全量 RaSP ΔΔG (优化版: DS预加载 + PDB分组缓存 + 断点续传)
已有结果: 492 OK, 断点续传
总:2179, 已完成:492, 待处理:1687

预加载 10 个 DownstreamModel...
  已加载 10 个
按 Uniprot 分组: 679 个 PDB

  101/1687 OK=96 速率=2.2var/s ETA=716s
  200/1687 OK=191 速率=2.3var/s ETA=635s
  302/1687 OK=293 速率=2.2var/s ETA=637s
  402/1687 OK=393 速率=2.2var/s ETA=584s
  500/1687 OK=491 速率=2.2var/s ETA=548s
  601/1687 OK=592 速率=2.2var/s ETA=492s
  701/1687 OK=692 速率=2.2var/s ETA=450s
  800/1687 OK=791 速率=2.1var/s ETA=414s
  900/1687 OK=891 速率=2.2var/s ETA=360s
  1002/1687 OK=993 速率=2.2var/s ETA=315s
  1100/1687 OK=1091 速率=2.1var/s ETA=280s
  1200/1687 OK=1189 速率=2.1var/s ETA=235s
  1301/1687 OK=1290 速率=2.0var/s ETA=190s
  1402/1687 OK=1391 速率=2.0var/s ETA=142s
  1502/1687 OK=1491 速率=2.0var/s ETA=93s
  1600/1687 OK=1589 速率=2.0var/s ETA=45s

完成! 耗时=875s 速率=1.9var/s
成功:2168/2179 (99.5%)
status
ok              2168
no_clean_pdb      11
ddg_rasp: mean=1.651 std=2.136 min=-2.068 max=11.041


## 9e. 加入 PCA(50) 模型评估 (ddg_rasp vs ddg_esm2)

**运行前提**: 9d 已跑完，`ddg_rasp.csv` 已生成。

同折对比三个模型:
1. v3 + PCA(50)
2. v3 + PCA(50) + ddg_esm2 (Task 4 序列 ΔΔG)
3. v3 + PCA(50) + ddg_rasp (本 Task 结构 ΔΔG)

In [7]:
RUN_EVAL = True  # ← 改为 True 后运行 (需先跑完 9d)

if RUN_EVAL:
    from sklearn.decomposition import PCA
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import roc_auc_score, average_precision_score
    from sklearn.utils.class_weight import compute_sample_weight
    from xgboost import XGBClassifier
    
    # 加载结构 ddg
    df_rasp = pd.read_csv(BASE_PATH + "ddg_rasp.csv")
    print(f"加载 ddg_rasp.csv: {len(df_rasp)} 行, OK={(df_rasp['status']=='ok').sum()}")
    
    # 映射到全量
    full_keys = (df_main["Gene"] + "||" + df_main["Mutation_used"]).tolist()
    ddg_map = dict(zip(df_rasp["KEY"], df_rasp["ddg_rasp"]))
    ddg_rasp_full = np.array([ddg_map.get(k, np.nan) for k in full_keys], dtype=np.float32)
    ddg_rasp_feat = ddg_rasp_full.reshape(-1, 1)
    
    # 加载 esm2 ddg (对比)
    try:
        df_esm2 = pd.read_csv(BASE_PATH + "ddg_esm2.csv")
        ddg_e_map = dict(zip(df_esm2["KEY"], df_esm2["ddg_esm2"]))
        ddg_esm2_full = np.array([ddg_e_map.get(k, np.nan) for k in full_keys], dtype=np.float32)
        ddg_esm2_feat = ddg_esm2_full.reshape(-1, 1)
        has_esm2 = True
    except:
        has_esm2 = False
        print("ddg_esm2.csv 未找到")
    
    # 特征加载
    df_feat = pd.read_csv(BASE_PATH + "features_v3.csv")
    ID_COLS = ["KEY", "Gene", "reloc_v3"]
    NAN_PLACEHOLDERS = ["ddg", "plddt_diff"]
    exclude_cols = ID_COLS + NAN_PLACEHOLDERS
    feat_cols = [c for c in df_feat.columns if c not in exclude_cols]
    esm_cols = [c for c in feat_cols if c.startswith("esm_")]
    struct_cols_present = ["plddt", "sasa", "rsa", "ss_helix", "ss_strand",
                           "ss_coil", "delta_hydrophobicity", "struct_status"]
    
    X_full = df_feat[feat_cols].values.astype(np.float32)
    esm_idx = [feat_cols.index(c) for c in esm_cols]
    struct_idx = [feat_cols.index(c) for c in struct_cols_present if c in feat_cols]
    X_esm = X_full[:, esm_idx]; X_struct = X_full[:, struct_idx]
    y_bin = (df_feat["reloc_v3"].values > 0).astype(int)
    groups = df_feat["Gene"].values
    
    # 同折 CV
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    N_COMP = 50
    
    oof_pca = np.zeros(len(y_bin), dtype=np.float32)
    oof_rasp = np.zeros(len(y_bin), dtype=np.float32)
    oof_esm2 = np.zeros(len(y_bin), dtype=np.float32) if has_esm2 else None
    
    for fold, (tr_idx, te_idx) in enumerate(cv.split(X_full, y_bin, groups)):
        y_tr = y_bin[tr_idx]; y_te = y_bin[te_idx]
        
        # PCA
        imp_e = SimpleImputer(strategy="median"); scl_e = StandardScaler()
        Xe_tr = scl_e.fit_transform(imp_e.fit_transform(X_esm[tr_idx])).astype(np.float32)
        Xe_te = scl_e.transform(imp_e.transform(X_esm[te_idx])).astype(np.float32)
        pca = PCA(n_components=N_COMP, random_state=42)
        Xe_tr_pca = pca.fit_transform(Xe_tr).astype(np.float32)
        Xe_te_pca = pca.transform(Xe_te).astype(np.float32)
        
        # Struct
        imp_s = SimpleImputer(strategy="median"); scl_s = StandardScaler()
        Xs_tr = scl_s.fit_transform(imp_s.fit_transform(X_struct[tr_idx])).astype(np.float32)
        Xs_te = scl_s.transform(imp_s.transform(X_struct[te_idx])).astype(np.float32)
        
        X_tr_pca = np.hstack([Xe_tr_pca, Xs_tr]).astype(np.float32)
        X_te_pca = np.hstack([Xe_te_pca, Xs_te]).astype(np.float32)
        sw = compute_sample_weight("balanced", y_tr)
        
        # 1) PCA only
        m = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.5,
                          objective="binary:logistic", eval_metric="aucpr",
                          random_state=42, n_jobs=-1, tree_method="hist")
        m.fit(X_tr_pca, y_tr, sample_weight=sw, verbose=False)
        oof_pca[te_idx] = m.predict_proba(X_te_pca)[:, 1]
        
        # 2) PCA + ddg_rasp
        imp_r = SimpleImputer(strategy="median")
        ddg_r_tr = imp_r.fit_transform(ddg_rasp_feat[tr_idx]).astype(np.float32)
        ddg_r_te = imp_r.transform(ddg_rasp_feat[te_idx]).astype(np.float32)
        m_r = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.5,
                            objective="binary:logistic", eval_metric="aucpr",
                            random_state=42, n_jobs=-1, tree_method="hist")
        m_r.fit(np.hstack([X_tr_pca, ddg_r_tr]).astype(np.float32), y_tr,
                sample_weight=sw, verbose=False)
        oof_rasp[te_idx] = m_r.predict_proba(
            np.hstack([X_te_pca, ddg_r_te]).astype(np.float32))[:, 1]
        
        # 3) PCA + ddg_esm2
        if has_esm2:
            imp_e2 = SimpleImputer(strategy="median")
            ddg_e_tr = imp_e2.fit_transform(ddg_esm2_feat[tr_idx]).astype(np.float32)
            ddg_e_te = imp_e2.transform(ddg_esm2_feat[te_idx]).astype(np.float32)
            m_e2 = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                 subsample=0.8, colsample_bytree=0.5,
                                 objective="binary:logistic", eval_metric="aucpr",
                                 random_state=42, n_jobs=-1, tree_method="hist")
            m_e2.fit(np.hstack([X_tr_pca, ddg_e_tr]).astype(np.float32), y_tr,
                     sample_weight=sw, verbose=False)
            oof_esm2[te_idx] = m_e2.predict_proba(
                np.hstack([X_te_pca, ddg_e_te]).astype(np.float32))[:, 1]
        
        a_p = roc_auc_score(y_te, oof_pca[te_idx])
        a_r = roc_auc_score(y_te, oof_rasp[te_idx])
        a_e = roc_auc_score(y_te, oof_esm2[te_idx]) if has_esm2 else 0
        print(f"  Fold {fold}: PCA={a_p:.4f}  PCA+ddg_rasp={a_r:.4f}  PCA+ddg_esm2={a_e:.4f}")
    
    # 汇总
    br = float(y_bin.sum() / len(y_bin))
    auroc_pca = roc_auc_score(y_bin, oof_pca)
    auroc_rasp = roc_auc_score(y_bin, oof_rasp)
    
    print(f"\n{'='*75}")
    print(f"  RaSP 结构 ΔΔG 最终对比")
    print(f"  n={len(y_bin)}, 正例={int(y_bin.sum())}, base_rate={br:.4f}")
    print(f"  {'模型':<40s} {'AUROC':>8s} {'Δ vs PCA':>10s}")
    print(f"  {'-'*58}")
    print(f"  {'v3 + PCA(50)':<40s} {auroc_pca:>8.4f}")
    print(f"  {'v3 + PCA(50) + ddg_rasp':<40s} {auroc_rasp:>8.4f} {auroc_rasp-auroc_pca:>+10.4f}")
    if has_esm2:
        auroc_esm2 = roc_auc_score(y_bin, oof_esm2)
        print(f"  {'v3 + PCA(50) + ddg_esm2':<40s} {auroc_esm2:>8.4f} {auroc_esm2-auroc_pca:>+10.4f}")
    print(f"{'='*75}")
else:
    print("RUN_EVAL = False。请先跑完 9d (全量计算) 后将 RUN_EVAL 改为 True。")

加载 ddg_rasp.csv: 2179 行, OK=2168
  Fold 0: PCA=0.6685  PCA+ddg_rasp=0.7036  PCA+ddg_esm2=0.6705
  Fold 1: PCA=0.6246  PCA+ddg_rasp=0.6318  PCA+ddg_esm2=0.6506
  Fold 2: PCA=0.5527  PCA+ddg_rasp=0.5386  PCA+ddg_esm2=0.5225
  Fold 3: PCA=0.6048  PCA+ddg_rasp=0.5932  PCA+ddg_esm2=0.6398
  Fold 4: PCA=0.6204  PCA+ddg_rasp=0.5979  PCA+ddg_esm2=0.5919

  RaSP 结构 ΔΔG 最终对比
  n=2179, 正例=235, base_rate=0.1078
  模型                                          AUROC   Δ vs PCA
  ----------------------------------------------------------
  v3 + PCA(50)                               0.6144
  v3 + PCA(50) + ddg_rasp                    0.6152    +0.0007
  v3 + PCA(50) + ddg_esm2                    0.6187    +0.0043
